In [4]:
# Run in a fresh Colab runtime.
!pip -q install uv

import subprocess
import sys
from pathlib import Path

ENV = Path("/content/alignscore-env")
PYTHON = str(ENV / "bin/python")

def run(command):
    subprocess.run(command, check=True)

# Create the venv securely using Python's built-in venv module instead of uv venv
run([sys.executable, "-m", "venv", str(ENV)])

run([
    "uv", "pip", "install", "--python", PYTHON,
    "torch==1.13.1",
    "numpy==1.24.4",
    "transformers==4.30.2",
    "pytorch-lightning==1.9.5",
    "torchmetrics==0.11.4",
    "spacy==3.7.5",
    "nltk==3.8.1",
    "scikit-learn==1.3.2",
    "pandas==2.0.3",
    "huggingface-hub==0.25.2",
    "protobuf==3.20.0",
    "tensorboard==2.14.0",
    "datasets==2.14.7",
    "pyarrow==14.0.2",
    "jsonlines==2.0.0",
    "tqdm",
])

run([
    "uv", "pip", "install", "--python", PYTHON,
    "--no-deps",
    "git+https://github.com/yuh-zha/AlignScore.git",
])

# Install the spaCy model directly using uv to avoid internal pip dependency issues
run([
    "uv", "pip", "install", "--python", PYTHON,
    "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.0/en_core_web_sm-3.7.0-py3-none-any.whl"
])

run([
    PYTHON, "-c",
    "import nltk; nltk.download('punkt', quiet=True)"
])

print("Installation complete.")

Installation complete.


In [5]:
import subprocess

RAG_DIR = Path("/content/RAGTruth")

if not RAG_DIR.exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/ParticleMedia/RAGTruth.git",
        str(RAG_DIR),
    ])

DATA_COMMIT = subprocess.check_output(
    ["git", "-C", str(RAG_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("RAGTruth commit:", DATA_COMMIT)
print("Dataset files:")
for path in RAG_DIR.rglob("*.jsonl"):
    print(path)

RAGTruth commit: c103204b9ce28d6bbad859304bf30de72b8ed8fe
Dataset files:
/content/RAGTruth/dataset/source_info.jsonl
/content/RAGTruth/dataset/response.jsonl


In [6]:
%%writefile /content/evaluate_alignscore.py
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from alignscore import AlignScore

# ---------- Configuration ----------
SEED = 42
N_VALIDATION = 50
N_TEST = 50             # Set to None to evaluate all eligible test responses.
BATCH_SIZE = 1
TASKS = {"QA", "Summary"}  # Start with text evidence; exclude Data2txt initially.
OUT = Path("/content/alignscore_results")
OUT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

# ---------- Load dataset ----------
def find_unique(filename):
    matches = list(Path("/content/RAGTruth").rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {filename}, found: {matches}")
    return matches[0]

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

sources = {
    str(x["source_id"]): x
    for x in read_jsonl(find_unique("source_info.jsonl"))
}
responses = read_jsonl(find_unique("response.jsonl"))

def evidence(source):
    info = source["source_info"]
    if source["task_type"] == "Summary":
        assert isinstance(info, str)
        return info
    if source["task_type"] == "QA":
        # The question is not treated as evidence.
        passages = info["passages"]
        assert isinstance(passages, str)
        return passages
    raise ValueError(source["task_type"])

rows = []
for response in responses:
    source_id = str(response["source_id"])
    source = sources[source_id]

    if source["task_type"] not in TASKS:
        continue
    # Explicitly limit this pilot to good-quality responses.
    if response["quality"] != "good":
        continue

    context = evidence(source)
    claim = response["response"]
    if not context.strip() or not claim.strip():
        continue

    rows.append({
        "id": str(response["id"]),
        "source_id": source_id,
        "split": response["split"],
        "task": source["task_type"],
        "generator": response["model"],
        "context": context,
        "response": claim,
        "gold": int(len(response["labels"]) > 0),
    })

df = pd.DataFrame(rows)
assert not df.empty

train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()
assert not train.empty and not test.empty

# Keep source documents separate across development and test.
assert set(train.source_id).isdisjoint(set(test.source_id))

# Create a validation pool from training sources, not test responses.
train_source_ids = sorted(train.source_id.unique())
_, val_source_ids = train_test_split(
    train_source_ids, test_size=0.20, random_state=SEED
)
validation = train[train.source_id.isin(val_source_ids)].copy()

def sample_rows(frame, n):
    if n is None or n >= len(frame):
        return frame.reset_index(drop=True)
    # Stratify by label when possible; retain approximate natural prevalence.
    if frame.gold.nunique() == 2 and frame.gold.value_counts().min() >= 2:
        selected, _ = train_test_split(
            frame,
            train_size=n,
            stratify=frame.gold,
            random_state=SEED,
        )
        return selected.reset_index(drop=True)
    return frame.sample(n=n, random_state=SEED).reset_index(drop=True)

validation = sample_rows(validation, N_VALIDATION)
test = sample_rows(test, N_TEST)

for name, frame in [("validation", validation), ("test", test)]:
    print(f"\n{name}: {len(frame)} responses")
    print(pd.crosstab(frame.task, frame.gold))
    frame.drop(columns=["context", "response"]).to_csv(
        OUT / f"{name}_selection.csv", index=False
    )

# ---------- Load pretrained AlignScore ----------
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("\nDevice:", device)

checkpoint = hf_hub_download(
    repo_id="yzha/AlignScore",
    filename="AlignScore-base.ckpt",
)

scorer = AlignScore(
    model="roberta-base",
    batch_size=BATCH_SIZE,
    device=device,
    ckpt_path=checkpoint,
    evaluation_mode="nli_sp",
)

# Smoke test: inspect these scores; do not use them for threshold selection.
print("\nSmoke-test consistency scores:")
print(scorer.score(
    contexts=[
        "The study enrolled 34 participants.",
        "The study enrolled 34 participants.",
    ],
    claims=[
        "The study enrolled 34 participants.",
        "The study enrolled 500 participants.",
    ],
))

# ---------- Score and save incrementally ----------
def score_frame(frame, name):
    path = OUT / f"{name}_predictions.csv"
    scored = []

    for row in tqdm(
        frame.to_dict("records"), desc=f"Scoring {name}"
    ):
        if device.startswith("cuda"):
            torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            consistency = float(scorer.score(
                contexts=[row["context"]],
                claims=[row["response"]],
            )[0])

        if device.startswith("cuda"):
            torch.cuda.synchronize()
        seconds = time.perf_counter() - start

        if not np.isfinite(consistency):
            raise RuntimeError(f"Non-finite score for {row['id']}")

        scored.append({
            **row,
            "consistency_score": consistency,
            "hallucination_score": 1.0 - consistency,
            "seconds": seconds,
        })
        # Save after every response.
        pd.DataFrame(scored).to_csv(path, index=False)

    return pd.DataFrame(scored)

val_scores = score_frame(validation, "validation")
test_scores = score_frame(test, "test")

# ---------- Select threshold using validation ONLY ----------
if val_scores.gold.nunique() < 2:
    raise RuntimeError(
        "Validation sample needs both labels. Increase N_VALIDATION."
    )

thresholds = np.unique(np.concatenate([
    [-1e-6],
    val_scores.hallucination_score.to_numpy(),
    [1.000001],
]))

def f1_at(threshold):
    predictions = (
        val_scores.hallucination_score.to_numpy() >= threshold
    ).astype(int)
    return precision_recall_fscore_support(
        val_scores.gold, predictions,
        average="binary", zero_division=0,
    )[2]

threshold = float(max(thresholds, key=f1_at))
print("\nValidation-selected threshold:", threshold)

# ---------- Evaluate held-out test sample ----------
def metrics(frame):
    y = frame.gold.to_numpy()
    scores = frame.hallucination_score.to_numpy()
    predicted = (scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, predicted, average="binary", zero_division=0
    )

    both_labels = len(np.unique(y)) == 2
    return {
        "n": len(frame),
        "hallucination_prevalence": float(y.mean()),
        "threshold": threshold,
        "auroc": float(roc_auc_score(y, scores)) if both_labels else None,
        "average_precision": (
            float(average_precision_score(y, scores))
            if both_labels else None
        ),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "mean_seconds_per_response": float(frame.seconds.mean()),
        "confusion_matrix_labels_0_1": confusion_matrix(
            y, predicted, labels=[0, 1]
        ).tolist(),
    }

results = {
    "overall": metrics(test_scores),
    "by_task": {
        task: metrics(group)
        for task, group in test_scores.groupby("task")
    },
    "settings": {
        "seed": SEED,
        "model": "AlignScore-base",
        "mode": "nli_sp",
        "device": device,
        "tasks": sorted(TASKS),
        "quality_filter": "good",
        "label_policy": "any annotated span, including implicit_true",
    },
}

test_scores["predicted"] = (
    test_scores.hallucination_score >= threshold
).astype(int)
test_scores.to_csv(OUT / "test_predictions.csv", index=False)
test_scores[test_scores.gold != test_scores.predicted].to_csv(
    OUT / "test_errors.csv", index=False
)

with open(OUT / "metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nTEST RESULTS")
print(json.dumps(results, indent=2))
print("\nFiles saved to:", OUT)

Writing /content/evaluate_alignscore.py


In [19]:
import subprocess
from pathlib import Path

# Define explicit paths
ENV = Path("/content/alignscore-env")
PYTHON = str(ENV / "bin/python")

# Install an older version of setuptools that contains pkg_resources
subprocess.run([
    "uv", "pip", "install", "--python", PYTHON, "--force-reinstall", "setuptools==65.5.0"
], check=True)

# Stream output while the script runs.
process = subprocess.Popen(
    [PYTHON, "-u", "/content/evaluate_alignscore.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="")

if process.wait() != 0:
    raise RuntimeError("Evaluation failed. Check the error printed above.")


validation: 50 responses
gold      0  1
task          
QA       13  9
Summary  21  7

test: 50 responses
gold      0  1
task          
QA       17  3
Summary  23  7

Device: cpu
/content/alignscore-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkp

In [20]:
import json
import pandas as pd
import shutil
from google.colab import files

RESULTS = Path("/content/alignscore_results")

with open(RESULTS / "metrics.json") as f:
    metrics = json.load(f)

display(pd.DataFrame([metrics["overall"]]))
display(pd.read_csv(RESULTS / "test_errors.csv").head())

# Record the dataset version and installed packages.
(RESULTS / "ragtruth_commit.txt").write_text(DATA_COMMIT)
with open(RESULTS / "environment.txt", "w") as f:
    subprocess.run(
        ["uv", "pip", "freeze", "--python", PYTHON],
        stdout=f, check=True,
    )

shutil.make_archive(
    "/content/alignscore_results", "zip", root_dir=RESULTS
)
files.download("/content/alignscore_results.zip")

,n,hallucination_prevalence,threshold,auroc,average_precision,precision,recall,f1,mean_seconds_per_response,confusion_matrix_labels_0_1
0,50,0.2,0.290311,0.83,0.581094,0.434783,1.0,0.606061,25.407891,"[[27, 13], [0, 10]]"


,id,source_id,split,task,generator,context,response,gold,consistency_score,hallucination_score,seconds,predicted
0,1456,15836,test,Summary,llama-2-13b-chat,"Istanbul, Turkey (CNN)Sitting on a sunny bench...",Here is a summary of the article in 200 words ...,0,0.613372,0.386628,65.577309,1
1,1354,15819,test,Summary,llama-2-13b-chat,If there was ever any doubt that the Obama adm...,Here is a summary of the article in 198 words:...,0,0.653700,0.346300,33.625593,1
2,14513,15339,test,QA,llama-2-70b-chat,passage 1:An ad hominem (Latin for to the man ...,The main difference between ad hominem fallacy...,0,0.432330,0.567670,7.083517,1
3,13133,15107,test,QA,llama-2-70b-chat,passage 1:3-Place brown sugar in a plastic con...,"Based on the given passages, here's how to sof...",0,0.616933,0.383067,31.994221,1
4,3884,13818,test,Summary,mistral-7B-instruct,Hannah Rose reveals why she quit Survivor in n...,"In the latest episode of Survivor 45, Hannah R...",0,0.320299,0.679701,40.758910,1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>